In [1]:
pip install --upgrade ultralytics

^C
Note: you may need to restart the kernel to use updated packages.


In [2]:
from ultralytics import YOLO
import torch

print("YOLO imported successfully!")
print("PyTorch version:", torch.__version__)

# Check and set up GPU
if torch.cuda.is_available():
    torch.cuda.set_device(0)
    torch.cuda.empty_cache()
    print(f" GPU detected: {torch.cuda.get_device_name(0)}")
    print(f"   VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
    device = 'cuda'
else:
    print("CUDA not available - falling back to CPU")
    print("   Install PyTorch with CUDA: pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118")
    device = 'cpu'

print(f"Using device: {device}")

YOLO imported successfully!
PyTorch version: 2.7.1+cu118
 GPU detected: NVIDIA GeForce RTX 2060
   VRAM: 6.4 GB
Using device: cuda


In [3]:
# train_RGB.ipynb
# Cell 1 — imports and sanity check

from ultralytics import YOLO
import torch, os

print(f"PyTorch version : {torch.__version__}")
print(f"CUDA available  : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU             : {torch.cuda.get_device_name(0)}")
    print(f"VRAM            : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

PyTorch version : 2.7.1+cu118
CUDA available  : True
GPU             : NVIDIA GeForce RTX 2060
VRAM            : 6.4 GB


In [5]:
import os

# Change to Desktop
os.chdir('C:/Users/DELL/Desktop/c')

# Verify
print("Current working directory:", os.getcwd())
print("\nContents of Desktop:")
print(os.listdir('.'))

Current working directory: C:\Users\DELL\Desktop\c

Contents of Desktop:
['Codes', 'conv.m', 'conversion - Copy.m', 'conversion.m', 'Copy_of_inspect_simd.m', 'inspect_simd.m', 'LGB_COMPLETE_results_nc15.txt', 'RGB_COMPLETE_results_nc15.txt', 'rgb_passthrough.m', 'rgb_to_lgb.m', 'rgb_to_lgbv2.m', 'rgb_to_oklab_full.m', 'rgb_to_ygb.m', 'runs', 'S3_verify_conversion.m', 'S4_build_datasets.m', 'SIMD_LGB', 'SIMD_LGB_test_nc15.yaml', 'SIMD_LGB_val_nc15.yaml', 'SIMD_RGB', 'SIMD_RGB_test_nc15.yaml', 'SIMD_RGB_val_nc15.yaml', 'SIMD_YGB', 'SIMD_YGB_test_nc15.yaml', 'SIMD_YGB_val_nc15.yaml', 'SSL.m', 'SSLconversion.m', 'SSL_LGB', 'SSL_RGB', 'SSL_test.m', 'SSL_train.m', 'SSL_val.m', 'SSL_YGB', 'YGB_COMPLETE_results_nc15.txt', 'yolo12n.pt', 'yolo26n.pt', 'yolov8n.pt']


In [10]:
import os

# Check what's in your SSL_RGB directory
dataset_path = 'SSL_RGB'
if os.path.exists(dataset_path):
    print("Contents of SSL_RGB:")
    for item in os.listdir(dataset_path):
        print(f"  - {item}")
        
    # Check if train/val directories exist
    for folder in ['train', 'val', 'test']:
        folder_path = os.path.join(dataset_path, folder)
        if os.path.exists(folder_path):
            print(f"\n{folder} contents:")
            for subitem in os.listdir(folder_path):
                print(f"  - {subitem}")
else:
    print(f"SSL_RGB directory not found at {dataset_path}")
    print(f"Current location: {os.getcwd()}")

Contents of SSL_RGB:
  - data.yaml
  - test
  - train
  - val

train contents:
  - images
  - labels

val contents:
  - images
  - labels

test contents:
  - images
  - labels


In [11]:
# Run this once before training
import os

cache_files = [
    'SSL_RGB/train/labels.cache',
    'SSL_RGB/val/labels.cache',
    'SSL_RGB/train/labels.cache.npy',
    'SSL_RGB/val/labels.cache.npy'
]

for cf in cache_files:
    if os.path.exists(cf):
        os.remove(cf)
        print(f"Removed: {cf}")
        
print("Cache cleared. Restart training and warnings should disappear.")

Cache cleared. Restart training and warnings should disappear.


## RGB  YOLOv8

In [14]:
import torch
import torch.nn as nn
from ultralytics import YOLO
from ultralytics.models.yolo.detect import DetectionTrainer

# ── Same weight computation as above ──────────────────────────────────────────
class_counts = [3927, 2706, 5455, 2948]
N_total      = 15036
C            = 4

raw_weights  = [N_total / (C * n) for n in class_counts]
max_w        = max(raw_weights)                          # ≈ 57.85
norm_weights = [w / max_w for w in raw_weights]

weights_tensor = torch.tensor(norm_weights, dtype=torch.float32)  # shape [4]

# ── Custom Trainer — injects Eq. (8) weighted BCE into YOLO loss ──────────────
class WeightedDetectionTrainer(DetectionTrainer):

    def get_model(self, cfg=None, weights=None, verbose=True):
        model = super().get_model(cfg=cfg, weights=weights, verbose=verbose)

        device = next(model.parameters()).device
        w      = weights_tensor.to(device)              # move to GPU

        # Replace BCEcls with per-class weighted version — implements Eq. (8):
        # L_cls = -Σ w̃_i [ y_i log(ŷ_i) + (1-y_i) log(1-ŷ_i) ]
        if hasattr(model, 'criterion') and model.criterion is not None:
            model.criterion.BCEcls = nn.BCEWithLogitsLoss(
                pos_weight = w,
                reduction  = 'none'
            )
            print(f"✅ Weighted BCE injected | max_w={max_w:.2f} | "
                  f"fighter w̃={norm_weights[9]:.4f} | "
                  f"car w̃={norm_weights[0]:.4f}")
        return model


# ── Training call — same hyperparams as your paper ────────────────────────────
model   = YOLO('yolov8n.pt')    
model.trainer = WeightedDetectionTrainer

results = model.train(
    data          = 'SSL_RGB/data.yaml',
    epochs        = 100,
    imgsz         = 640,
    batch         = 8,            # paper: batch=8
    lr0           = 0.1,          # paper: lr0=0.1
    optimizer     = 'SGD',
    momentum      = 0.937,
    weight_decay  = 0.0005,
    warmup_epochs = 3,
    cos_lr        = True,         # paper: cosine annealing
    augment       = True,
    pretrained    = False,        # paper: trained from scratch
    patience      = 15,           # paper: early stopping patience=15
    device        = 0,
    project       = 'runs',
    name          = 'SSL_RGB_YOLOv8n_weighted',
    exist_ok      = True,
    seed          = 42,
    workers       = 4,
    amp           = False,
    resume        = False,
    cache         = False,
)

print(f"\nBest weights: {results.save_dir}/weights/best.pt")

Ultralytics 8.4.162  Python-3.13.9 torch-2.7.1+cu118 CUDA:0 (NVIDIA GeForce RTX 2060, 6144MiB)
engine\trainer: agnostic_nms=False, amp=False, angle=1.0, augment=True, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=SSL_RGB/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=100, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.1, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=SSL_RGB_YOLOv8n_weighted, nbs=64, nms=None, opset=

In [15]:
from ultralytics import YOLO
import yaml
import os
import shutil

# RGB paths
base_path = r"C:\Users\DELL\Desktop\c\SSL_RGB"
model_path = r'C:\Users\DELL\Desktop\c\runs\detect\runs\SSL_RGB_YOLOv8n_weighted\weights\best.pt'

# DELETE OLD CACHE FILES
print("Cleaning old cache files...")
cache_files = [
    os.path.join(base_path, "val", "labels.cache"),
    os.path.join(base_path, "test", "labels.cache"),
]
for cache_file in cache_files:
    if os.path.exists(cache_file):
        os.remove(cache_file)
        print(f"✓ Deleted: {cache_file}")

# CORRECT class configuration (4 classes)
num_classes = 4
class_names = ['airplane', 'boat', 'car', 'ship']

# Create VAL YAML
val_yaml_path = 'SSL_RGB_val_nc4.yaml'
val_yaml_content = {
    'path': base_path,
    'train': 'train/images',
    'val': 'VAL/images',
    'test': 'test/images',
    'nc': num_classes,
    'names': class_names,
}
with open(val_yaml_path, 'w') as f:
    yaml.dump(val_yaml_content, f, default_flow_style=False)
print(f"✓ Created: {val_yaml_path}")

# Create TEST YAML
test_yaml_path = 'SSL_RGB_test_nc4.yaml'
test_yaml_content = {
    'path': base_path,
    'train': 'train/images',
    'val': 'VAL/images',
    'test': 'test/images',
    'nc': num_classes,
    'names': class_names,
}
with open(test_yaml_path, 'w') as f:
    yaml.dump(test_yaml_content, f, default_flow_style=False)
print(f"✓ Created: {test_yaml_path}")

# Load model
model = YOLO(model_path)

# Run VALIDATION
print("\n" + "="*50)
print("RUNNING VALIDATION ON RGB")
print("="*50)
val_results = model.val(
    data=val_yaml_path,
    split='val',
    imgsz=640,
    batch=16,
    device=0,
    workers=0,
    verbose=True
)

print("\n" + "="*50)
print("RGB VALIDATION RESULTS (nc=4)")
print("="*50)
print(f"mAP50-95: {val_results.box.map:.4f}")
print(f"mAP50:     {val_results.box.map50:.4f}")
print(f"mAP75:     {val_results.box.map75:.4f}")
print(f"Precision: {val_results.box.mp:.4f}")
print(f"Recall:    {val_results.box.mr:.4f}")

# FIXED PER-CLASS RESULTS
print("\n" + "="*50)
print("PER-CLASS RESULTS (mAP50)")
print("="*50)
for i, name in enumerate(class_names):
    if i < len(val_results.box.ap50):  # Fixed: check length instead of using index()
        print(f"{name:20} mAP50: {val_results.box.ap50[i]:.4f}")

# Run TESTING
print("\n" + "="*50)
print("RUNNING TESTING ON RGB")
print("="*50)
test_results = model.val(
    data=test_yaml_path,
    split='test',
    imgsz=640,
    batch=16,
    device=0,
    workers=0,
    verbose=True
)

print("\n" + "="*50)
print("RGB TEST RESULTS (nc=4)")
print("="*50)
print(f"mAP50-95: {test_results.box.map:.4f}")
print(f"mAP50:     {test_results.box.map50:.4f}")
print(f"mAP75:     {test_results.box.map75:.4f}")
print(f"Precision: {test_results.box.mp:.4f}")
print(f"Recall:    {test_results.box.mr:.4f}")

# Save results
with open('RGB_COMPLETE_results_nc4.txt', 'w') as f:
    f.write("RGB MODEL - COMPLETE RESULTS (CORRECT nc=4)\n")
    f.write("=" * 50 + "\n\n")
    
    f.write("VALIDATION RESULTS\n")
    f.write("-" * 20 + "\n")
    f.write(f"mAP50-95: {val_results.box.map:.4f}\n")
    f.write(f"mAP50:     {val_results.box.map50:.4f}\n")
    f.write(f"mAP75:     {val_results.box.map75:.4f}\n")
    f.write(f"Precision: {val_results.box.mp:.4f}\n")
    f.write(f"Recall:    {val_results.box.mr:.4f}\n\n")
    
    f.write("TEST RESULTS\n")
    f.write("-" * 20 + "\n")
    f.write(f"mAP50-95: {test_results.box.map:.4f}\n")
    f.write(f"mAP50:     {test_results.box.map50:.4f}\n")
    f.write(f"mAP75:     {test_results.box.map75:.4f}\n")
    f.write(f"Precision: {test_results.box.mp:.4f}\n")
    f.write(f"Recall:    {test_results.box.mr:.4f}\n")

print("\n✓ Results saved to 'RGB_COMPLETE_results_nc4.txt'")
print("\n" + "="*50)
print("RGB COMPLETE!")
print("="*50)

Cleaning old cache files...
✓ Deleted: C:\Users\DELL\Desktop\c\SSL_RGB\val\labels.cache
✓ Created: SSL_RGB_val_nc4.yaml
✓ Created: SSL_RGB_test_nc4.yaml

RUNNING VALIDATION ON RGB
Ultralytics 8.4.162  Python-3.13.9 torch-2.7.1+cu118 CUDA:0 (NVIDIA GeForce RTX 2060, 6144MiB)
Model summary (fused): 72 layers, 3,006,428 parameters, 0 gradients, 8.1 GFLOPs
val: Fast image access  (ping: 0.00.0 ms, read: 1489.3909.6 MB/s, size: 175.8 KB)
val: Scanning C:\Users\DELL\Desktop\c\SSL_RGB\val\labels... 132 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 132/132 2.2Kit/s 0.1s
val: New cache created: C:\Users\DELL\Desktop\c\SSL_RGB\val\labels.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 9/9 4.6it/s 2.0s0.2s
                   all        132       1992        0.8      0.681      0.767      0.437
               class_0         37        367      0.885      0.817      0.886      0.469
               class_1         33        657 

## YGB  YOLOv8

In [16]:
import torch
import torch.nn as nn
from ultralytics import YOLO
from ultralytics.models.yolo.detect import DetectionTrainer

# ── Same weight computation as above ──────────────────────────────────────────
class_counts = [3927, 2706, 5455, 2948]
N_total      = 15036
C            = 4

raw_weights  = [N_total / (C * n) for n in class_counts]
max_w        = max(raw_weights)                          # ≈ 57.85
norm_weights = [w / max_w for w in raw_weights]


weights_tensor = torch.tensor(norm_weights, dtype=torch.float32)  # shape [4]

# ── Custom Trainer — injects Eq. (8) weighted BCE into YOLO loss ──────────────
class WeightedDetectionTrainer(DetectionTrainer):

    def get_model(self, cfg=None, weights=None, verbose=True):
        model = super().get_model(cfg=cfg, weights=weights, verbose=verbose)

        device = next(model.parameters()).device
        w      = weights_tensor.to(device)              # move to GPU

        # Replace BCEcls with per-class weighted version — implements Eq. (8):
        # L_cls = -Σ w̃_i [ y_i log(ŷ_i) + (1-y_i) log(1-ŷ_i) ]
        if hasattr(model, 'criterion') and model.criterion is not None:
            model.criterion.BCEcls = nn.BCEWithLogitsLoss(
                pos_weight = w,
                reduction  = 'none'
            )
            print(f"✅ Weighted BCE injected | max_w={max_w:.2f} | "
                  f"fighter w̃={norm_weights[9]:.4f} | "
                  f"car w̃={norm_weights[0]:.4f}")
        return model


# ── Training call — same hyperparams as your paper ────────────────────────────
model   = YOLO('yolov8n.pt')    
model.trainer = WeightedDetectionTrainer

results = model.train(
    data          = 'SSL_YGB/data.yaml',
    epochs        = 100,
    imgsz         = 640,
    batch         = 8,            # paper: batch=8
    lr0           = 0.1,          # paper: lr0=0.1
    optimizer     = 'SGD',
    momentum      = 0.937,
    weight_decay  = 0.0005,
    warmup_epochs = 3,
    cos_lr        = True,         # paper: cosine annealing
    augment       = True,
    pretrained    = False,        # paper: trained from scratch
    patience      = 15,           # paper: early stopping patience=15
    device        = 0,
    project       = 'runs',
    name          = 'SSL_YGB_YOLOv8n_weighted',
    exist_ok      = True,
    seed          = 42,
    workers       = 4,
    amp           = False,
    resume        = False,
    cache         = False,
)

print(f"\nBest weights: {results.save_dir}/weights/best.pt")

Ultralytics 8.4.162  Python-3.13.9 torch-2.7.1+cu118 CUDA:0 (NVIDIA GeForce RTX 2060, 6144MiB)
engine\trainer: agnostic_nms=False, amp=False, angle=1.0, augment=True, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=SSL_YGB/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=100, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.1, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=SSL_YGB_YOLOv8n_weighted, nbs=64, nms=None, opset=

In [17]:
from ultralytics import YOLO
import yaml
import os
import shutil

# RGB paths
base_path = r"C:\Users\DELL\Desktop\c\SSL_YGB"
model_path = r'C:\Users\DELL\Desktop\c\runs\detect\runs\SSL_YGB_YOLOv8n_weighted\weights\best.pt'

# DELETE OLD CACHE FILES
print("Cleaning old cache files...")
cache_files = [
    os.path.join(base_path, "val", "labels.cache"),
    os.path.join(base_path, "test", "labels.cache"),
]
for cache_file in cache_files:
    if os.path.exists(cache_file):
        os.remove(cache_file)
        print(f"✓ Deleted: {cache_file}")

# CORRECT class configuration (4 classes)
num_classes = 4
class_names = ['airplane', 'boat', 'car', 'ship']

# Create VAL YAML
val_yaml_path = 'SSL_YGB_val_nc4.yaml'
val_yaml_content = {
    'path': base_path,
    'train': 'train/images',
    'val': 'VAL/images',
    'test': 'test/images',
    'nc': num_classes,
    'names': class_names,
}
with open(val_yaml_path, 'w') as f:
    yaml.dump(val_yaml_content, f, default_flow_style=False)
print(f"✓ Created: {val_yaml_path}")

# Create TEST YAML
test_yaml_path = 'SSL_YGB_test_nc4.yaml'
test_yaml_content = {
    'path': base_path,
    'train': 'train/images',
    'val': 'VAL/images',
    'test': 'test/images',
    'nc': num_classes,
    'names': class_names,
}
with open(test_yaml_path, 'w') as f:
    yaml.dump(test_yaml_content, f, default_flow_style=False)
print(f"✓ Created: {test_yaml_path}")

# Load model
model = YOLO(model_path)

# Run VALIDATION
print("\n" + "="*50)
print("RUNNING VALIDATION ON YGB")
print("="*50)
val_results = model.val(
    data=val_yaml_path,
    split='val',
    imgsz=640,
    batch=16,
    device=0,
    workers=0,
    verbose=True
)

print("\n" + "="*50)
print("YGB VALIDATION RESULTS (nc=4)")
print("="*50)
print(f"mAP50-95: {val_results.box.map:.4f}")
print(f"mAP50:     {val_results.box.map50:.4f}")
print(f"mAP75:     {val_results.box.map75:.4f}")
print(f"Precision: {val_results.box.mp:.4f}")
print(f"Recall:    {val_results.box.mr:.4f}")

# FIXED PER-CLASS RESULTS
print("\n" + "="*50)
print("PER-CLASS RESULTS (mAP50)")
print("="*50)
for i, name in enumerate(class_names):
    if i < len(val_results.box.ap50):  # Fixed: check length instead of using index()
        print(f"{name:20} mAP50: {val_results.box.ap50[i]:.4f}")

# Run TESTING
print("\n" + "="*50)
print("RUNNING TESTING ON YGB")
print("="*50)
test_results = model.val(
    data=test_yaml_path,
    split='test',
    imgsz=640,
    batch=16,
    device=0,
    workers=0,
    verbose=True
)

print("\n" + "="*50)
print("YGB TEST RESULTS (nc=4)")
print("="*50)
print(f"mAP50-95: {test_results.box.map:.4f}")
print(f"mAP50:     {test_results.box.map50:.4f}")
print(f"mAP75:     {test_results.box.map75:.4f}")
print(f"Precision: {test_results.box.mp:.4f}")
print(f"Recall:    {test_results.box.mr:.4f}")

# Save results
with open('YGB_COMPLETE_results_nc4.txt', 'w') as f:
    f.write("YGB MODEL - COMPLETE RESULTS (CORRECT nc=4)\n")
    f.write("=" * 50 + "\n\n")
    
    f.write("VALIDATION RESULTS\n")
    f.write("-" * 20 + "\n")
    f.write(f"mAP50-95: {val_results.box.map:.4f}\n")
    f.write(f"mAP50:     {val_results.box.map50:.4f}\n")
    f.write(f"mAP75:     {val_results.box.map75:.4f}\n")
    f.write(f"Precision: {val_results.box.mp:.4f}\n")
    f.write(f"Recall:    {val_results.box.mr:.4f}\n\n")
    
    f.write("TEST RESULTS\n")
    f.write("-" * 20 + "\n")
    f.write(f"mAP50-95: {test_results.box.map:.4f}\n")
    f.write(f"mAP50:     {test_results.box.map50:.4f}\n")
    f.write(f"mAP75:     {test_results.box.map75:.4f}\n")
    f.write(f"Precision: {test_results.box.mp:.4f}\n")
    f.write(f"Recall:    {test_results.box.mr:.4f}\n")

print("\n✓ Results saved to 'YGB_COMPLETE_results_nc4.txt'")
print("\n" + "="*50)
print("YGB COMPLETE!")
print("="*50)

Cleaning old cache files...
✓ Deleted: C:\Users\DELL\Desktop\c\SSL_YGB\val\labels.cache
✓ Created: SSL_YGB_val_nc4.yaml
✓ Created: SSL_YGB_test_nc4.yaml

RUNNING VALIDATION ON YGB
Ultralytics 8.4.162  Python-3.13.9 torch-2.7.1+cu118 CUDA:0 (NVIDIA GeForce RTX 2060, 6144MiB)
Model summary (fused): 72 layers, 3,006,428 parameters, 0 gradients, 8.1 GFLOPs
val: Fast image access  (ping: 0.00.0 ms, read: 2284.01247.0 MB/s, size: 330.2 KB)
val: Scanning C:\Users\DELL\Desktop\c\SSL_YGB\val\labels... 132 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 132/132 2.6Kit/s 0.1s
val: New cache created: C:\Users\DELL\Desktop\c\SSL_YGB\val\labels.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 9/9 4.4it/s 2.1s0.2s
                   all        132       1992      0.786      0.711      0.772      0.435
               class_0         37        367      0.867       0.82      0.891      0.466
               class_1         33        657

## LGB  YOLOv8

In [18]:
import torch
import torch.nn as nn
from ultralytics import YOLO
from ultralytics.models.yolo.detect import DetectionTrainer

# ── Same weight computation as above ──────────────────────────────────────────
class_counts = [3927, 2706, 5455, 2948]
N_total      = 15036
C            = 4

raw_weights  = [N_total / (C * n) for n in class_counts]
max_w        = max(raw_weights)                          # ≈ 57.85
norm_weights = [w / max_w for w in raw_weights]


weights_tensor = torch.tensor(norm_weights, dtype=torch.float32)  # shape [4]

# ── Custom Trainer — injects Eq. (8) weighted BCE into YOLO loss ──────────────
class WeightedDetectionTrainer(DetectionTrainer):

    def get_model(self, cfg=None, weights=None, verbose=True):
        model = super().get_model(cfg=cfg, weights=weights, verbose=verbose)

        device = next(model.parameters()).device
        w      = weights_tensor.to(device)              # move to GPU

        # Replace BCEcls with per-class weighted version — implements Eq. (8):
        # L_cls = -Σ w̃_i [ y_i log(ŷ_i) + (1-y_i) log(1-ŷ_i) ]
        if hasattr(model, 'criterion') and model.criterion is not None:
            model.criterion.BCEcls = nn.BCEWithLogitsLoss(
                pos_weight = w,
                reduction  = 'none'
            )
            print(f"✅ Weighted BCE injected | max_w={max_w:.2f} | "
                  f"fighter w̃={norm_weights[9]:.4f} | "
                  f"car w̃={norm_weights[0]:.4f}")
        return model


# ── Training call — same hyperparams as your paper ────────────────────────────
model   = YOLO('yolov8n.pt')    # or 'yolo26n.pt'
model.trainer = WeightedDetectionTrainer

results = model.train(
    data          = 'SSL_LGB/data.yaml',
    epochs        = 100,
    imgsz         = 640,
    batch         = 8,            # paper: batch=8
    lr0           = 0.1,          # paper: lr0=0.1
    optimizer     = 'SGD',
    momentum      = 0.937,
    weight_decay  = 0.0005,
    warmup_epochs = 3,
    cos_lr        = True,         # paper: cosine annealing
    augment       = True,
    pretrained    = False,        # paper: trained from scratch
    patience      = 15,           # paper: early stopping patience=15
    device        = 0,
    project       = 'runs',
    name          = 'SSL_LGB_YOLOv8n_weighted',
    exist_ok      = True,
    seed          = 42,
    workers       = 4,
    amp           = False,
    resume        = False,
    cache         = False,
)

print(f"\nBest weights: {results.save_dir}/weights/best.pt")

Ultralytics 8.4.162  Python-3.13.9 torch-2.7.1+cu118 CUDA:0 (NVIDIA GeForce RTX 2060, 6144MiB)
engine\trainer: agnostic_nms=False, amp=False, angle=1.0, augment=True, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=SSL_LGB/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=100, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.1, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=SSL_LGB_YOLOv8n_weighted, nbs=64, nms=None, opset=

In [19]:
from ultralytics import YOLO
import yaml
import os
import shutil

# RGB paths
base_path = r"C:\Users\DELL\Desktop\c\SSL_LGB"
model_path = r'C:\Users\DELL\Desktop\c\runs\detect\runs\SSL_LGB_YOLOv8n_weighted\weights\best.pt'

# DELETE OLD CACHE FILES
print("Cleaning old cache files...")
cache_files = [
    os.path.join(base_path, "val", "labels.cache"),
    os.path.join(base_path, "test", "labels.cache"),
]
for cache_file in cache_files:
    if os.path.exists(cache_file):
        os.remove(cache_file)
        print(f"✓ Deleted: {cache_file}")

# CORRECT class configuration (4 classes)
num_classes = 4
class_names = ['airplane', 'boat', 'car', 'ship']

# Create VAL YAML
val_yaml_path = 'SSL_LGB_val_nc4.yaml'
val_yaml_content = {
    'path': base_path,
    'train': 'train/images',
    'val': 'VAL/images',
    'test': 'test/images',
    'nc': num_classes,
    'names': class_names,
}
with open(val_yaml_path, 'w') as f:
    yaml.dump(val_yaml_content, f, default_flow_style=False)
print(f"✓ Created: {val_yaml_path}")

# Create TEST YAML
test_yaml_path = 'SSL_LGB_test_nc4.yaml'
test_yaml_content = {
    'path': base_path,
    'train': 'train/images',
    'val': 'VAL/images',
    'test': 'test/images',
    'nc': num_classes,
    'names': class_names,
}
with open(test_yaml_path, 'w') as f:
    yaml.dump(test_yaml_content, f, default_flow_style=False)
print(f"✓ Created: {test_yaml_path}")

# Load model
model = YOLO(model_path)

# Run VALIDATION
print("\n" + "="*50)
print("RUNNING VALIDATION ON LGB")
print("="*50)
val_results = model.val(
    data=val_yaml_path,
    split='val',
    imgsz=640,
    batch=16,
    device=0,
    workers=0,
    verbose=True
)

print("\n" + "="*50)
print("LGB VALIDATION RESULTS (nc=4)")
print("="*50)
print(f"mAP50-95: {val_results.box.map:.4f}")
print(f"mAP50:     {val_results.box.map50:.4f}")
print(f"mAP75:     {val_results.box.map75:.4f}")
print(f"Precision: {val_results.box.mp:.4f}")
print(f"Recall:    {val_results.box.mr:.4f}")

# FIXED PER-CLASS RESULTS
print("\n" + "="*50)
print("PER-CLASS RESULTS (mAP50)")
print("="*50)
for i, name in enumerate(class_names):
    if i < len(val_results.box.ap50):  # Fixed: check length instead of using index()
        print(f"{name:20} mAP50: {val_results.box.ap50[i]:.4f}")

# Run TESTING
print("\n" + "="*50)
print("RUNNING TESTING ON LGB")
print("="*50)
test_results = model.val(
    data=test_yaml_path,
    split='test',
    imgsz=640,
    batch=16,
    device=0,
    workers=0,
    verbose=True
)

print("\n" + "="*50)
print("LGB TEST RESULTS (nc=4)")
print("="*50)
print(f"mAP50-95: {test_results.box.map:.4f}")
print(f"mAP50:     {test_results.box.map50:.4f}")
print(f"mAP75:     {test_results.box.map75:.4f}")
print(f"Precision: {test_results.box.mp:.4f}")
print(f"Recall:    {test_results.box.mr:.4f}")

# Save results
with open('LGB_COMPLETE_results_nc4.txt', 'w') as f:
    f.write("LGB MODEL - COMPLETE RESULTS (CORRECT nc=4)\n")
    f.write("=" * 50 + "\n\n")
    
    f.write("VALIDATION RESULTS\n")
    f.write("-" * 20 + "\n")
    f.write(f"mAP50-95: {val_results.box.map:.4f}\n")
    f.write(f"mAP50:     {val_results.box.map50:.4f}\n")
    f.write(f"mAP75:     {val_results.box.map75:.4f}\n")
    f.write(f"Precision: {val_results.box.mp:.4f}\n")
    f.write(f"Recall:    {val_results.box.mr:.4f}\n\n")
    
    f.write("TEST RESULTS\n")
    f.write("-" * 20 + "\n")
    f.write(f"mAP50-95: {test_results.box.map:.4f}\n")
    f.write(f"mAP50:     {test_results.box.map50:.4f}\n")
    f.write(f"mAP75:     {test_results.box.map75:.4f}\n")
    f.write(f"Precision: {test_results.box.mp:.4f}\n")
    f.write(f"Recall:    {test_results.box.mr:.4f}\n")

print("\n✓ Results saved to 'LGB_COMPLETE_results_nc4.txt'")
print("\n" + "="*50)
print("LGB COMPLETE!")
print("="*50)

Cleaning old cache files...
✓ Deleted: C:\Users\DELL\Desktop\c\SSL_LGB\val\labels.cache
✓ Created: SSL_LGB_val_nc4.yaml
✓ Created: SSL_LGB_test_nc4.yaml

RUNNING VALIDATION ON LGB
Ultralytics 8.4.162  Python-3.13.9 torch-2.7.1+cu118 CUDA:0 (NVIDIA GeForce RTX 2060, 6144MiB)
Model summary (fused): 72 layers, 3,006,428 parameters, 0 gradients, 8.1 GFLOPs
val: Fast image access  (ping: 0.00.0 ms, read: 2061.41110.4 MB/s, size: 336.1 KB)
val: Scanning C:\Users\DELL\Desktop\c\SSL_LGB\val\labels... 132 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 132/132 2.6Kit/s 0.1s
val: New cache created: C:\Users\DELL\Desktop\c\SSL_LGB\val\labels.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 9/9 4.4it/s 2.0s0.2s
                   all        132       1992      0.815      0.696      0.764      0.441
               class_0         37        367       0.94      0.798      0.896      0.485
               class_1         33        657

## RGB YOLOv12

In [20]:
import torch
import torch.nn as nn
from ultralytics import YOLO
from ultralytics.models.yolo.detect import DetectionTrainer

# ── Same weight computation as above ──────────────────────────────────────────
class_counts = [3927, 2706, 5455, 2948]
N_total      = 15036
C            = 4

raw_weights  = [N_total / (C * n) for n in class_counts]
max_w        = max(raw_weights)                          # ≈ 57.85
norm_weights = [w / max_w for w in raw_weights]

weights_tensor = torch.tensor(norm_weights, dtype=torch.float32)  # shape [4]

# ── Custom Trainer — injects Eq. (8) weighted BCE into YOLO loss ──────────────
class WeightedDetectionTrainer(DetectionTrainer):

    def get_model(self, cfg=None, weights=None, verbose=True):
        model = super().get_model(cfg=cfg, weights=weights, verbose=verbose)

        device = next(model.parameters()).device
        w      = weights_tensor.to(device)              # move to GPU

        # Replace BCEcls with per-class weighted version — implements Eq. (8):
        # L_cls = -Σ w̃_i [ y_i log(ŷ_i) + (1-y_i) log(1-ŷ_i) ]
        if hasattr(model, 'criterion') and model.criterion is not None:
            model.criterion.BCEcls = nn.BCEWithLogitsLoss(
                pos_weight = w,
                reduction  = 'none'
            )
            print(f"✅ Weighted BCE injected | max_w={max_w:.2f} | "
                  f"fighter w̃={norm_weights[9]:.4f} | "
                  f"car w̃={norm_weights[0]:.4f}")
        return model


# ── Training call — same hyperparams as your paper ────────────────────────────
model   = YOLO('yolo12n.pt')    # or 'yolo26n.pt'
model.trainer = WeightedDetectionTrainer

results = model.train(
    data          = 'SSL_RGB/data.yaml',
    epochs        = 100,
    imgsz         = 640,
    batch         = 8,            # paper: batch=8
    lr0           = 0.1,          # paper: lr0=0.1
    optimizer     = 'SGD',
    momentum      = 0.937,
    weight_decay  = 0.0005,
    warmup_epochs = 3,
    cos_lr        = True,         # paper: cosine annealing
    augment       = True,
    pretrained    = False,        # paper: trained from scratch
    patience      = 15,           # paper: early stopping patience=15
    device        = 0,
    project       = 'runs',
    name          = 'SSL_RGB_YOLOv12n_weighted',
    exist_ok      = True,
    seed          = 42,
    workers       = 4,
    amp           = False,
    resume        = False,
    cache         = False,
)

print(f"\nBest weights: {results.save_dir}/weights/best.pt")

Ultralytics 8.4.162  Python-3.13.9 torch-2.7.1+cu118 CUDA:0 (NVIDIA GeForce RTX 2060, 6144MiB)
engine\trainer: agnostic_nms=False, amp=False, angle=1.0, augment=True, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=SSL_RGB/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=100, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.1, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo12n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=SSL_RGB_YOLOv12n_weighted, nbs=64, nms=None, opset

In [25]:
from ultralytics import YOLO
import yaml
import os
import shutil

# RGB paths
base_path = r"C:\Users\DELL\Desktop\c\SSL_RGB"
model_path = r'C:\Users\DELL\Desktop\c\runs\detect\runs\SSL_RGB_YOLOv12n_weighted\weights\best.pt'

# DELETE OLD CACHE FILES
print("Cleaning old cache files...")
cache_files = [
    os.path.join(base_path, "val", "labels.cache"),
    os.path.join(base_path, "test", "labels.cache"),
]
for cache_file in cache_files:
    if os.path.exists(cache_file):
        os.remove(cache_file)
        print(f"✓ Deleted: {cache_file}")

# CORRECT class configuration (4 classes)
num_classes = 4
class_names = ['airplane', 'boat', 'car', 'ship']

# Create VAL YAML
val_yaml_path = 'SSL_RGB_val_nc4.yaml'
val_yaml_content = {
    'path': base_path,
    'train': 'train/images',
    'val': 'VAL/images',
    'test': 'test/images',
    'nc': num_classes,
    'names': class_names,
}
with open(val_yaml_path, 'w') as f:
    yaml.dump(val_yaml_content, f, default_flow_style=False)
print(f"✓ Created: {val_yaml_path}")

# Create TEST YAML
test_yaml_path = 'SSL_RGB_test_nc4.yaml'
test_yaml_content = {
    'path': base_path,
    'train': 'train/images',
    'val': 'VAL/images',
    'test': 'test/images',
    'nc': num_classes,
    'names': class_names,
}
with open(test_yaml_path, 'w') as f:
    yaml.dump(test_yaml_content, f, default_flow_style=False)
print(f"✓ Created: {test_yaml_path}")

# Load model
model = YOLO(model_path)

# Run VALIDATION
print("\n" + "="*50)
print("RUNNING VALIDATION ON RGB")
print("="*50)
val_results = model.val(
    data=val_yaml_path,
    split='val',
    imgsz=640,
    batch=16,
    device=0,
    workers=0,
    verbose=True
)

print("\n" + "="*50)
print("RGB VALIDATION RESULTS (nc=4)")
print("="*50)
print(f"mAP50-95: {val_results.box.map:.4f}")
print(f"mAP50:     {val_results.box.map50:.4f}")
print(f"mAP75:     {val_results.box.map75:.4f}")
print(f"Precision: {val_results.box.mp:.4f}")
print(f"Recall:    {val_results.box.mr:.4f}")

# FIXED PER-CLASS RESULTS
print("\n" + "="*50)
print("PER-CLASS RESULTS (mAP50)")
print("="*50)
for i, name in enumerate(class_names):
    if i < len(val_results.box.ap50):  # Fixed: check length instead of using index()
        print(f"{name:20} mAP50: {val_results.box.ap50[i]:.4f}")

# Run TESTING
print("\n" + "="*50)
print("RUNNING TESTING ON RGB")
print("="*50)
test_results = model.val(
    data=test_yaml_path,
    split='test',
    imgsz=640,
    batch=16,
    device=0,
    workers=0,
    verbose=True
)

print("\n" + "="*50)
print("RGB TEST RESULTS (nc=4)")
print("="*50)
print(f"mAP50-95: {test_results.box.map:.4f}")
print(f"mAP50:     {test_results.box.map50:.4f}")
print(f"mAP75:     {test_results.box.map75:.4f}")
print(f"Precision: {test_results.box.mp:.4f}")
print(f"Recall:    {test_results.box.mr:.4f}")

# Save results
with open('RGB_COMPLETE_results_nc4.txt', 'w') as f:
    f.write("RGB MODEL - COMPLETE RESULTS (CORRECT nc=4)\n")
    f.write("=" * 50 + "\n\n")
    
    f.write("VALIDATION RESULTS\n")
    f.write("-" * 20 + "\n")
    f.write(f"mAP50-95: {val_results.box.map:.4f}\n")
    f.write(f"mAP50:     {val_results.box.map50:.4f}\n")
    f.write(f"mAP75:     {val_results.box.map75:.4f}\n")
    f.write(f"Precision: {val_results.box.mp:.4f}\n")
    f.write(f"Recall:    {val_results.box.mr:.4f}\n\n")
    
    f.write("TEST RESULTS\n")
    f.write("-" * 20 + "\n")
    f.write(f"mAP50-95: {test_results.box.map:.4f}\n")
    f.write(f"mAP50:     {test_results.box.map50:.4f}\n")
    f.write(f"mAP75:     {test_results.box.map75:.4f}\n")
    f.write(f"Precision: {test_results.box.mp:.4f}\n")
    f.write(f"Recall:    {test_results.box.mr:.4f}\n")


Cleaning old cache files...
✓ Created: SSL_RGB_val_nc4.yaml
✓ Created: SSL_RGB_test_nc4.yaml

RUNNING VALIDATION ON RGB
Ultralytics 8.4.162  Python-3.13.9 torch-2.7.1+cu118 CUDA:0 (NVIDIA GeForce RTX 2060, 6144MiB)
YOLOv12n summary (fused): 159 layers, 2,557,508 parameters, 0 gradients, 7.3 GFLOPs
val: Fast image access  (ping: 0.00.0 ms, read: 1252.4860.0 MB/s, size: 175.8 KB)
val: Scanning C:\Users\DELL\Desktop\c\SSL_RGB\val\labels... 132 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 132/132 2.2Kit/s 0.1s
val: New cache created: C:\Users\DELL\Desktop\c\SSL_RGB\val\labels.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 9/9 4.6it/s 1.9s0.2s
                   all        132       1992      0.729      0.623      0.687      0.371
               class_0         37        367      0.824      0.826      0.872      0.472
               class_1         33        657       0.71      0.524       0.62      0.306
            

## YGB YOLOv12

In [26]:
import torch
import torch.nn as nn
from ultralytics import YOLO
from ultralytics.models.yolo.detect import DetectionTrainer

# ── Same weight computation as above ──────────────────────────────────────────
class_counts = [3927, 2706, 5455, 2948]
N_total      = 15036
C            = 4

raw_weights  = [N_total / (C * n) for n in class_counts]
max_w        = max(raw_weights)                          # ≈ 57.85
norm_weights = [w / max_w for w in raw_weights]

weights_tensor = torch.tensor(norm_weights, dtype=torch.float32)  # shape [4]

# ── Custom Trainer — injects Eq. (8) weighted BCE into YOLO loss ──────────────
class WeightedDetectionTrainer(DetectionTrainer):

    def get_model(self, cfg=None, weights=None, verbose=True):
        model = super().get_model(cfg=cfg, weights=weights, verbose=verbose)

        device = next(model.parameters()).device
        w      = weights_tensor.to(device)              # move to GPU

        # Replace BCEcls with per-class weighted version — implements Eq. (8):
        # L_cls = -Σ w̃_i [ y_i log(ŷ_i) + (1-y_i) log(1-ŷ_i) ]
        if hasattr(model, 'criterion') and model.criterion is not None:
            model.criterion.BCEcls = nn.BCEWithLogitsLoss(
                pos_weight = w,
                reduction  = 'none'
            )
            print(f"✅ Weighted BCE injected | max_w={max_w:.2f} | "
                  f"fighter w̃={norm_weights[9]:.4f} | "
                  f"car w̃={norm_weights[0]:.4f}")
        return model


# ── Training call — same hyperparams as your paper ────────────────────────────
model   = YOLO('yolo12n.pt')    # or 'yolo26n.pt'
model.trainer = WeightedDetectionTrainer

results = model.train(
    data          = 'SSL_YGB/data.yaml',
    epochs        = 100,
    imgsz         = 640,
    batch         = 8,            # paper: batch=8
    lr0           = 0.1,          # paper: lr0=0.1
    optimizer     = 'SGD',
    momentum      = 0.937,
    weight_decay  = 0.0005,
    warmup_epochs = 3,
    cos_lr        = True,         # paper: cosine annealing
    augment       = True,
    pretrained    = False,        # paper: trained from scratch
    patience      = 15,           # paper: early stopping patience=15
    device        = 0,
    project       = 'runs',
    name          = 'SSL_YGB_YOLOv12n_weighted',
    exist_ok      = True,
    seed          = 42,
    workers       = 4,
    amp           = False,
    resume        = False,
    cache         = False,
)

print(f"\nBest weights: {results.save_dir}/weights/best.pt")

New https://pypi.org/project/ultralytics/8.4.163 available  Update with 'pip install -U ultralytics'
Ultralytics 8.4.162  Python-3.13.9 torch-2.7.1+cu118 CUDA:0 (NVIDIA GeForce RTX 2060, 6144MiB)
engine\trainer: agnostic_nms=False, amp=False, angle=1.0, augment=True, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=SSL_YGB/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=100, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.1, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo12n.pt,

In [30]:
from ultralytics import YOLO
import yaml
import os
import shutil

# RGB paths
base_path = r"C:\Users\DELL\Desktop\c\SSL_YGB"
model_path = r'C:\Users\DELL\Desktop\c\runs\detect\runs\SSL_YGB_YOLOv12n_weighted\weights\best.pt'

# DELETE OLD CACHE FILES
print("Cleaning old cache files...")
cache_files = [
    os.path.join(base_path, "val", "labels.cache"),
    os.path.join(base_path, "test", "labels.cache"),
]
for cache_file in cache_files:
    if os.path.exists(cache_file):
        os.remove(cache_file)
        print(f"✓ Deleted: {cache_file}")

# CORRECT class configuration (4 classes)
num_classes = 4
class_names = ['airplane', 'boat', 'car', 'ship']

# Create VAL YAML
val_yaml_path = 'SSL_YGB_val_nc4.yaml'
val_yaml_content = {
    'path': base_path,
    'train': 'train/images',
    'val': 'VAL/images',
    'test': 'test/images',
    'nc': num_classes,
    'names': class_names,
}
with open(val_yaml_path, 'w') as f:
    yaml.dump(val_yaml_content, f, default_flow_style=False)
print(f"✓ Created: {val_yaml_path}")

# Create TEST YAML
test_yaml_path = 'SSL_YGB_test_nc4.yaml'
test_yaml_content = {
    'path': base_path,
    'train': 'train/images',
    'val': 'VAL/images',
    'test': 'test/images',
    'nc': num_classes,
    'names': class_names,
}
with open(test_yaml_path, 'w') as f:
    yaml.dump(test_yaml_content, f, default_flow_style=False)
print(f"✓ Created: {test_yaml_path}")

# Load model
model = YOLO(model_path)

# Run VALIDATION
print("\n" + "="*50)
print("RUNNING VALIDATION ON YGB")
print("="*50)
val_results = model.val(
    data=val_yaml_path,
    split='val',
    imgsz=640,
    batch=16,
    device=0,
    workers=0,
    verbose=True
)

print("\n" + "="*50)
print("YGB VALIDATION RESULTS (nc=4)")
print("="*50)
print(f"mAP50-95: {val_results.box.map:.4f}")
print(f"mAP50:     {val_results.box.map50:.4f}")
print(f"mAP75:     {val_results.box.map75:.4f}")
print(f"Precision: {val_results.box.mp:.4f}")
print(f"Recall:    {val_results.box.mr:.4f}")

# FIXED PER-CLASS RESULTS
print("\n" + "="*50)
print("PER-CLASS RESULTS (mAP50)")
print("="*50)
for i, name in enumerate(class_names):
    if i < len(val_results.box.ap50):  # Fixed: check length instead of using index()
        print(f"{name:20} mAP50: {val_results.box.ap50[i]:.4f}")

# Run TESTING
print("\n" + "="*50)
print("RUNNING TESTING ON YGB")
print("="*50)
test_results = model.val(
    data=test_yaml_path,
    split='test',
    imgsz=640,
    batch=16,
    device=0,
    workers=0,
    verbose=True
)

print("\n" + "="*50)
print("YGB TEST RESULTS (nc=4)")
print("="*50)
print(f"mAP50-95: {test_results.box.map:.4f}")
print(f"mAP50:     {test_results.box.map50:.4f}")
print(f"mAP75:     {test_results.box.map75:.4f}")
print(f"Precision: {test_results.box.mp:.4f}")
print(f"Recall:    {test_results.box.mr:.4f}")

# Save results
with open('YGB_COMPLETE_results_nc4.txt', 'w') as f:
    f.write("YGB MODEL - COMPLETE RESULTS (CORRECT nc=4)\n")
    f.write("=" * 50 + "\n\n")
    
    f.write("VALIDATION RESULTS\n")
    f.write("-" * 20 + "\n")
    f.write(f"mAP50-95: {val_results.box.map:.4f}\n")
    f.write(f"mAP50:     {val_results.box.map50:.4f}\n")
    f.write(f"mAP75:     {val_results.box.map75:.4f}\n")
    f.write(f"Precision: {val_results.box.mp:.4f}\n")
    f.write(f"Recall:    {val_results.box.mr:.4f}\n\n")
    
    f.write("TEST RESULTS\n")
    f.write("-" * 20 + "\n")
    f.write(f"mAP50-95: {test_results.box.map:.4f}\n")
    f.write(f"mAP50:     {test_results.box.map50:.4f}\n")
    f.write(f"mAP75:     {test_results.box.map75:.4f}\n")
    f.write(f"Precision: {test_results.box.mp:.4f}\n")
    f.write(f"Recall:    {test_results.box.mr:.4f}\n")


Cleaning old cache files...
✓ Created: SSL_YGB_val_nc4.yaml
✓ Created: SSL_YGB_test_nc4.yaml

RUNNING VALIDATION ON YGB
Ultralytics 8.4.162  Python-3.13.9 torch-2.7.1+cu118 CUDA:0 (NVIDIA GeForce RTX 2060, 6144MiB)
YOLOv12n summary (fused): 159 layers, 2,557,508 parameters, 0 gradients, 7.3 GFLOPs
val: Fast image access  (ping: 0.00.0 ms, read: 2471.41394.2 MB/s, size: 330.2 KB)
val: Scanning C:\Users\DELL\Desktop\c\SSL_YGB\val\labels... 132 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 132/132 2.2Kit/s 0.1s
val: New cache created: C:\Users\DELL\Desktop\c\SSL_YGB\val\labels.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 9/9 4.5it/s 2.0s0.2s
                   all        132       1992      0.741      0.603      0.682      0.371
               class_0         37        367      0.885      0.775      0.858      0.455
               class_1         33        657      0.749      0.508      0.622      0.315
           

## LGB YOLOv12

In [31]:
import torch
import torch.nn as nn
from ultralytics import YOLO
from ultralytics.models.yolo.detect import DetectionTrainer

# ── Same weight computation as above ──────────────────────────────────────────
class_counts = [3927, 2706, 5455, 2948]
N_total      = 15036
C            = 4

raw_weights  = [N_total / (C * n) for n in class_counts]
max_w        = max(raw_weights)                          # ≈ 57.85
norm_weights = [w / max_w for w in raw_weights]

weights_tensor = torch.tensor(norm_weights, dtype=torch.float32)  # shape [4]

# ── Custom Trainer — injects Eq. (8) weighted BCE into YOLO loss ──────────────
class WeightedDetectionTrainer(DetectionTrainer):

    def get_model(self, cfg=None, weights=None, verbose=True):
        model = super().get_model(cfg=cfg, weights=weights, verbose=verbose)

        device = next(model.parameters()).device
        w      = weights_tensor.to(device)              # move to GPU

        # Replace BCEcls with per-class weighted version — implements Eq. (8):
        # L_cls = -Σ w̃_i [ y_i log(ŷ_i) + (1-y_i) log(1-ŷ_i) ]
        if hasattr(model, 'criterion') and model.criterion is not None:
            model.criterion.BCEcls = nn.BCEWithLogitsLoss(
                pos_weight = w,
                reduction  = 'none'
            )
            print(f"✅ Weighted BCE injected | max_w={max_w:.2f} | "
                  f"fighter w̃={norm_weights[9]:.4f} | "
                  f"car w̃={norm_weights[0]:.4f}")
        return model


# ── Training call — same hyperparams as your paper ────────────────────────────
model   = YOLO('yolo12n.pt')    # or 'yolo26n.pt'
model.trainer = WeightedDetectionTrainer

results = model.train(
    data          = 'SSL_LGB/data.yaml',
    epochs        = 100,
    imgsz         = 640,
    batch         = 8,            # paper: batch=8
    lr0           = 0.1,          # paper: lr0=0.1
    optimizer     = 'SGD',
    momentum      = 0.937,
    weight_decay  = 0.0005,
    warmup_epochs = 3,
    cos_lr        = True,         # paper: cosine annealing
    augment       = True,
    pretrained    = False,        # paper: trained from scratch
    patience      = 15,           # paper: early stopping patience=15
    device        = 0,
    project       = 'runs',
    name          = 'SSL_LGB_YOLOv12n_weighted',
    exist_ok      = True,
    seed          = 42,
    workers       = 4,
    amp           = False,
    resume        = False,
    cache         = False,
)

print(f"\nBest weights: {results.save_dir}/weights/best.pt")

New https://pypi.org/project/ultralytics/8.4.163 available  Update with 'pip install -U ultralytics'
Ultralytics 8.4.162  Python-3.13.9 torch-2.7.1+cu118 CUDA:0 (NVIDIA GeForce RTX 2060, 6144MiB)
engine\trainer: agnostic_nms=False, amp=False, angle=1.0, augment=True, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=SSL_LGB/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=100, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.1, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo12n.pt,

In [32]:
from ultralytics import YOLO
import yaml
import os
import shutil

# RGB paths
base_path = r"C:\Users\DELL\Desktop\c\SSL_LGB"
model_path = r'C:\Users\DELL\Desktop\c\runs\detect\runs\SSL_LGB_YOLOv12n_weighted\weights\best.pt'

# DELETE OLD CACHE FILES
print("Cleaning old cache files...")
cache_files = [
    os.path.join(base_path, "val", "labels.cache"),
    os.path.join(base_path, "test", "labels.cache"),
]
for cache_file in cache_files:
    if os.path.exists(cache_file):
        os.remove(cache_file)
        print(f"✓ Deleted: {cache_file}")

# CORRECT class configuration (4 classes)
num_classes = 4
class_names = ['airplane', 'boat', 'car', 'ship']

# Create VAL YAML
val_yaml_path = 'SSL_LGB_val_nc4.yaml'
val_yaml_content = {
    'path': base_path,
    'train': 'train/images',
    'val': 'VAL/images',
    'test': 'test/images',
    'nc': num_classes,
    'names': class_names,
}
with open(val_yaml_path, 'w') as f:
    yaml.dump(val_yaml_content, f, default_flow_style=False)
print(f"✓ Created: {val_yaml_path}")

# Create TEST YAML
test_yaml_path = 'SSL_LGB_test_nc4.yaml'
test_yaml_content = {
    'path': base_path,
    'train': 'train/images',
    'val': 'VAL/images',
    'test': 'test/images',
    'nc': num_classes,
    'names': class_names,
}
with open(test_yaml_path, 'w') as f:
    yaml.dump(test_yaml_content, f, default_flow_style=False)
print(f"✓ Created: {test_yaml_path}")

# Load model
model = YOLO(model_path)

# Run VALIDATION
print("\n" + "="*50)
print("RUNNING VALIDATION ON LGB")
print("="*50)
val_results = model.val(
    data=val_yaml_path,
    split='val',
    imgsz=640,
    batch=16,
    device=0,
    workers=0,
    verbose=True
)

print("\n" + "="*50)
print("LGB VALIDATION RESULTS (nc=4)")
print("="*50)
print(f"mAP50-95: {val_results.box.map:.4f}")
print(f"mAP50:     {val_results.box.map50:.4f}")
print(f"mAP75:     {val_results.box.map75:.4f}")
print(f"Precision: {val_results.box.mp:.4f}")
print(f"Recall:    {val_results.box.mr:.4f}")

# FIXED PER-CLASS RESULTS
print("\n" + "="*50)
print("PER-CLASS RESULTS (mAP50)")
print("="*50)
for i, name in enumerate(class_names):
    if i < len(val_results.box.ap50):  # Fixed: check length instead of using index()
        print(f"{name:20} mAP50: {val_results.box.ap50[i]:.4f}")

# Run TESTING
print("\n" + "="*50)
print("RUNNING TESTING ON LGB")
print("="*50)
test_results = model.val(
    data=test_yaml_path,
    split='test',
    imgsz=640,
    batch=16,
    device=0,
    workers=0,
    verbose=True
)

print("\n" + "="*50)
print("LGB TEST RESULTS (nc=4)")
print("="*50)
print(f"mAP50-95: {test_results.box.map:.4f}")
print(f"mAP50:     {test_results.box.map50:.4f}")
print(f"mAP75:     {test_results.box.map75:.4f}")
print(f"Precision: {test_results.box.mp:.4f}")
print(f"Recall:    {test_results.box.mr:.4f}")

# Save results
with open('LGB_COMPLETE_results_nc15.txt', 'w') as f:
    f.write("LGB MODEL - COMPLETE RESULTS (CORRECT nc=4)\n")
    f.write("=" * 50 + "\n\n")
    
    f.write("VALIDATION RESULTS\n")
    f.write("-" * 20 + "\n")
    f.write(f"mAP50-95: {val_results.box.map:.4f}\n")
    f.write(f"mAP50:     {val_results.box.map50:.4f}\n")
    f.write(f"mAP75:     {val_results.box.map75:.4f}\n")
    f.write(f"Precision: {val_results.box.mp:.4f}\n")
    f.write(f"Recall:    {val_results.box.mr:.4f}\n\n")
    
    f.write("TEST RESULTS\n")
    f.write("-" * 20 + "\n")
    f.write(f"mAP50-95: {test_results.box.map:.4f}\n")
    f.write(f"mAP50:     {test_results.box.map50:.4f}\n")
    f.write(f"mAP75:     {test_results.box.map75:.4f}\n")
    f.write(f"Precision: {test_results.box.mp:.4f}\n")
    f.write(f"Recall:    {test_results.box.mr:.4f}\n")


Cleaning old cache files...
✓ Deleted: C:\Users\DELL\Desktop\c\SSL_LGB\val\labels.cache
✓ Deleted: C:\Users\DELL\Desktop\c\SSL_LGB\test\labels.cache
✓ Created: SSL_LGB_val_nc4.yaml
✓ Created: SSL_LGB_test_nc4.yaml

RUNNING VALIDATION ON LGB
Ultralytics 8.4.162  Python-3.13.9 torch-2.7.1+cu118 CUDA:0 (NVIDIA GeForce RTX 2060, 6144MiB)
YOLOv12n summary (fused): 159 layers, 2,557,508 parameters, 0 gradients, 7.3 GFLOPs
val: Fast image access  (ping: 0.00.0 ms, read: 2110.41240.8 MB/s, size: 336.1 KB)
val: Scanning C:\Users\DELL\Desktop\c\SSL_LGB\val\labels... 132 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 132/132 2.3Kit/s 0.1s
val: New cache created: C:\Users\DELL\Desktop\c\SSL_LGB\val\labels.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 9/9 4.1it/s 2.2s0.2s
                   all        132       1992      0.774       0.62      0.712       0.39
               class_0         37        367      0.884      0.768  

## RGB  YOLOv26

In [33]:
import torch
import torch.nn as nn
from ultralytics import YOLO
from ultralytics.models.yolo.detect import DetectionTrainer

# ── Same weight computation as above ──────────────────────────────────────────
class_counts = [3927, 2706, 5455, 2948]
N_total      = 15036
C            = 4

raw_weights  = [N_total / (C * n) for n in class_counts]
max_w        = max(raw_weights)                          # ≈ 57.85
norm_weights = [w / max_w for w in raw_weights]

weights_tensor = torch.tensor(norm_weights, dtype=torch.float32)  # shape [4]

# ── Custom Trainer — injects Eq. (8) weighted BCE into YOLO loss ──────────────
class WeightedDetectionTrainer(DetectionTrainer):

    def get_model(self, cfg=None, weights=None, verbose=True):
        model = super().get_model(cfg=cfg, weights=weights, verbose=verbose)

        device = next(model.parameters()).device
        w      = weights_tensor.to(device)              # move to GPU

        # Replace BCEcls with per-class weighted version — implements Eq. (8):
        # L_cls = -Σ w̃_i [ y_i log(ŷ_i) + (1-y_i) log(1-ŷ_i) ]
        if hasattr(model, 'criterion') and model.criterion is not None:
            model.criterion.BCEcls = nn.BCEWithLogitsLoss(
                pos_weight = w,
                reduction  = 'none'
            )
            print(f"✅ Weighted BCE injected | max_w={max_w:.2f} | "
                  f"fighter w̃={norm_weights[9]:.4f} | "
                  f"car w̃={norm_weights[0]:.4f}")
        return model


# ── Training call — same hyperparams as your paper ────────────────────────────
model   = YOLO('yolo26n.pt')    # or 'yolo26n.pt'
model.trainer = WeightedDetectionTrainer

results = model.train(
    data          = 'SSL_RGB/data.yaml',
    epochs        = 100,
    imgsz         = 640,
    batch         = 8,            # paper: batch=8
    lr0           = 0.1,          # paper: lr0=0.1
    optimizer     = 'SGD',
    momentum      = 0.937,
    weight_decay  = 0.0005,
    warmup_epochs = 3,
    cos_lr        = True,         # paper: cosine annealing
    augment       = True,
    pretrained    = False,        # paper: trained from scratch
    patience      = 15,           # paper: early stopping patience=15
    device        = 0,
    project       = 'runs',
    name          = 'SSL_RGB_YOLO26n_weighted',
    exist_ok      = True,
    seed          = 42,
    workers       = 4,
    amp           = False,
    resume        = False,
    cache         = False,
)

print(f"\nBest weights: {results.save_dir}/weights/best.pt")

New https://pypi.org/project/ultralytics/8.4.163 available  Update with 'pip install -U ultralytics'
Ultralytics 8.4.162  Python-3.13.9 torch-2.7.1+cu118 CUDA:0 (NVIDIA GeForce RTX 2060, 6144MiB)
engine\trainer: agnostic_nms=False, amp=False, angle=1.0, augment=True, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=SSL_RGB/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=100, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.1, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo26n.pt,

In [34]:
from ultralytics import YOLO
import yaml
import os
import shutil

# RGB paths
base_path = r"C:\Users\DELL\Desktop\c\SSL_RGB"
model_path = r'C:\Users\DELL\Desktop\c\runs\detect\runs\SSL_RGB_YOLO26n_weighted\weights\best.pt'

# DELETE OLD CACHE FILES
print("Cleaning old cache files...")
cache_files = [
    os.path.join(base_path, "val", "labels.cache"),
    os.path.join(base_path, "test", "labels.cache"),
]
for cache_file in cache_files:
    if os.path.exists(cache_file):
        os.remove(cache_file)
        print(f"✓ Deleted: {cache_file}")

# CORRECT class configuration (4 classes)
num_classes = 4
class_names = ['airplane', 'boat', 'car', 'ship']

# Create VAL YAML
val_yaml_path = 'SSL_RGB_val_nc4.yaml'
val_yaml_content = {
    'path': base_path,
    'train': 'train/images',
    'val': 'VAL/images',
    'test': 'test/images',
    'nc': num_classes,
    'names': class_names,
}
with open(val_yaml_path, 'w') as f:
    yaml.dump(val_yaml_content, f, default_flow_style=False)
print(f"✓ Created: {val_yaml_path}")

# Create TEST YAML
test_yaml_path = 'SSL_RGB_test_nc4.yaml'
test_yaml_content = {
    'path': base_path,
    'train': 'train/images',
    'val': 'VAL/images',
    'test': 'test/images',
    'nc': num_classes,
    'names': class_names,
}
with open(test_yaml_path, 'w') as f:
    yaml.dump(test_yaml_content, f, default_flow_style=False)
print(f"✓ Created: {test_yaml_path}")

# Load model
model = YOLO(model_path)

# Run VALIDATION
print("\n" + "="*50)
print("RUNNING VALIDATION ON RGB")
print("="*50)
val_results = model.val(
    data=val_yaml_path,
    split='val',
    imgsz=640,
    batch=16,
    device=0,
    workers=0,
    verbose=True
)

print("\n" + "="*50)
print("RGB VALIDATION RESULTS (nc=4)")
print("="*50)
print(f"mAP50-95: {val_results.box.map:.4f}")
print(f"mAP50:     {val_results.box.map50:.4f}")
print(f"mAP75:     {val_results.box.map75:.4f}")
print(f"Precision: {val_results.box.mp:.4f}")
print(f"Recall:    {val_results.box.mr:.4f}")

# FIXED PER-CLASS RESULTS
print("\n" + "="*50)
print("PER-CLASS RESULTS (mAP50)")
print("="*50)
for i, name in enumerate(class_names):
    if i < len(val_results.box.ap50):  # Fixed: check length instead of using index()
        print(f"{name:20} mAP50: {val_results.box.ap50[i]:.4f}")

# Run TESTING
print("\n" + "="*50)
print("RUNNING TESTING ON RGB")
print("="*50)
test_results = model.val(
    data=test_yaml_path,
    split='test',
    imgsz=640,
    batch=16,
    device=0,
    workers=0,
    verbose=True
)

print("\n" + "="*50)
print("RGB TEST RESULTS (nc=4)")
print("="*50)
print(f"mAP50-95: {test_results.box.map:.4f}")
print(f"mAP50:     {test_results.box.map50:.4f}")
print(f"mAP75:     {test_results.box.map75:.4f}")
print(f"Precision: {test_results.box.mp:.4f}")
print(f"Recall:    {test_results.box.mr:.4f}")

# Save results
with open('RGB_COMPLETE_results_nc15.txt', 'w') as f:
    f.write("RGB MODEL - COMPLETE RESULTS (CORRECT nc=4)\n")
    f.write("=" * 50 + "\n\n")
    
    f.write("VALIDATION RESULTS\n")
    f.write("-" * 20 + "\n")
    f.write(f"mAP50-95: {val_results.box.map:.4f}\n")
    f.write(f"mAP50:     {val_results.box.map50:.4f}\n")
    f.write(f"mAP75:     {val_results.box.map75:.4f}\n")
    f.write(f"Precision: {val_results.box.mp:.4f}\n")
    f.write(f"Recall:    {val_results.box.mr:.4f}\n\n")
    
    f.write("TEST RESULTS\n")
    f.write("-" * 20 + "\n")
    f.write(f"mAP50-95: {test_results.box.map:.4f}\n")
    f.write(f"mAP50:     {test_results.box.map50:.4f}\n")
    f.write(f"mAP75:     {test_results.box.map75:.4f}\n")
    f.write(f"Precision: {test_results.box.mp:.4f}\n")
    f.write(f"Recall:    {test_results.box.mr:.4f}\n")


Cleaning old cache files...
✓ Deleted: C:\Users\DELL\Desktop\c\SSL_RGB\val\labels.cache
✓ Deleted: C:\Users\DELL\Desktop\c\SSL_RGB\test\labels.cache
✓ Created: SSL_RGB_val_nc4.yaml
✓ Created: SSL_RGB_test_nc4.yaml

RUNNING VALIDATION ON RGB
Ultralytics 8.4.162  Python-3.13.9 torch-2.7.1+cu118 CUDA:0 (NVIDIA GeForce RTX 2060, 6144MiB)
YOLO26n summary (fused): 120 layers, 2,375,616 parameters, 0 gradients, 5.3 GFLOPs
val: Fast image access  (ping: 0.00.0 ms, read: 1497.0958.7 MB/s, size: 175.8 KB)
val: Scanning C:\Users\DELL\Desktop\c\SSL_RGB\val\labels... 132 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 132/132 2.2Kit/s 0.1s
val: New cache created: C:\Users\DELL\Desktop\c\SSL_RGB\val\labels.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 9/9 4.5it/s 2.0s0.2s
                   all        132       1992      0.746      0.608      0.674      0.329
               class_0         37        367      0.907      0.782    

## YGB  YOLOv26

In [35]:
import torch
import torch.nn as nn
from ultralytics import YOLO
from ultralytics.models.yolo.detect import DetectionTrainer

# ── Same weight computation as above ──────────────────────────────────────────
class_counts = [3927, 2706, 5455, 2948]
N_total      = 15036
C            = 4

raw_weights  = [N_total / (C * n) for n in class_counts]
max_w        = max(raw_weights)                          # ≈ 57.85
norm_weights = [w / max_w for w in raw_weights]

weights_tensor = torch.tensor(norm_weights, dtype=torch.float32)  # shape [4]

# ── Custom Trainer — injects Eq. (8) weighted BCE into YOLO loss ──────────────
class WeightedDetectionTrainer(DetectionTrainer):

    def get_model(self, cfg=None, weights=None, verbose=True):
        model = super().get_model(cfg=cfg, weights=weights, verbose=verbose)

        device = next(model.parameters()).device
        w      = weights_tensor.to(device)              # move to GPU

        # Replace BCEcls with per-class weighted version — implements Eq. (8):
        # L_cls = -Σ w̃_i [ y_i log(ŷ_i) + (1-y_i) log(1-ŷ_i) ]
        if hasattr(model, 'criterion') and model.criterion is not None:
            model.criterion.BCEcls = nn.BCEWithLogitsLoss(
                pos_weight = w,
                reduction  = 'none'
            )
            print(f"✅ Weighted BCE injected | max_w={max_w:.2f} | "
                  f"fighter w̃={norm_weights[9]:.4f} | "
                  f"car w̃={norm_weights[0]:.4f}")
        return model


# ── Training call — same hyperparams as your paper ────────────────────────────
model   = YOLO('yolo26n.pt')    # or 'yolo26n.pt'
model.trainer = WeightedDetectionTrainer

results = model.train(
    data          = 'SSL_YGB/data.yaml',
    epochs        = 100,
    imgsz         = 640,
    batch         = 8,            # paper: batch=8
    lr0           = 0.1,          # paper: lr0=0.1
    optimizer     = 'SGD',
    momentum      = 0.937,
    weight_decay  = 0.0005,
    warmup_epochs = 3,
    cos_lr        = True,         # paper: cosine annealing
    augment       = True,
    pretrained    = False,        # paper: trained from scratch
    patience      = 15,           # paper: early stopping patience=15
    device        = 0,
    project       = 'runs',
    name          = 'SSL_YGB_YOLO26n_weighted',
    exist_ok      = True,
    seed          = 42,
    workers       = 4,
    amp           = False,
    resume        = False,
    cache         = False,
)

print(f"\nBest weights: {results.save_dir}/weights/best.pt")

New https://pypi.org/project/ultralytics/8.4.163 available  Update with 'pip install -U ultralytics'
Ultralytics 8.4.162  Python-3.13.9 torch-2.7.1+cu118 CUDA:0 (NVIDIA GeForce RTX 2060, 6144MiB)
engine\trainer: agnostic_nms=False, amp=False, angle=1.0, augment=True, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=SSL_YGB/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=100, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.1, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo26n.pt,

In [36]:
from ultralytics import YOLO
import yaml
import os
import shutil

# RGB paths
base_path = r"C:\Users\DELL\Desktop\c\SSL_YGB"
model_path = r'C:\Users\DELL\Desktop\c\runs\detect\runs\SSL_YGB_YOLO26n_weighted\weights\best.pt'

# DELETE OLD CACHE FILES
print("Cleaning old cache files...")
cache_files = [
    os.path.join(base_path, "val", "labels.cache"),
    os.path.join(base_path, "test", "labels.cache"),
]
for cache_file in cache_files:
    if os.path.exists(cache_file):
        os.remove(cache_file)
        print(f"✓ Deleted: {cache_file}")

# CORRECT class configuration (4 classes)
num_classes = 4
class_names = ['airplane', 'boat', 'car', 'ship']

# Create VAL YAML
val_yaml_path = 'SSL_YGB_val_nc4.yaml'
val_yaml_content = {
    'path': base_path,
    'train': 'train/images',
    'val': 'VAL/images',
    'test': 'test/images',
    'nc': num_classes,
    'names': class_names,
}
with open(val_yaml_path, 'w') as f:
    yaml.dump(val_yaml_content, f, default_flow_style=False)
print(f"✓ Created: {val_yaml_path}")

# Create TEST YAML
test_yaml_path = 'SSL_YGB_test_nc4.yaml'
test_yaml_content = {
    'path': base_path,
    'train': 'train/images',
    'val': 'VAL/images',
    'test': 'test/images',
    'nc': num_classes,
    'names': class_names,
}
with open(test_yaml_path, 'w') as f:
    yaml.dump(test_yaml_content, f, default_flow_style=False)
print(f"✓ Created: {test_yaml_path}")

# Load model
model = YOLO(model_path)

# Run VALIDATION
print("\n" + "="*50)
print("RUNNING VALIDATION ON YGB")
print("="*50)
val_results = model.val(
    data=val_yaml_path,
    split='val',
    imgsz=640,
    batch=16,
    device=0,
    workers=0,
    verbose=True
)

print("\n" + "="*50)
print("YGB VALIDATION RESULTS (nc=4)")
print("="*50)
print(f"mAP50-95: {val_results.box.map:.4f}")
print(f"mAP50:     {val_results.box.map50:.4f}")
print(f"mAP75:     {val_results.box.map75:.4f}")
print(f"Precision: {val_results.box.mp:.4f}")
print(f"Recall:    {val_results.box.mr:.4f}")

# FIXED PER-CLASS RESULTS
print("\n" + "="*50)
print("PER-CLASS RESULTS (mAP50)")
print("="*50)
for i, name in enumerate(class_names):
    if i < len(val_results.box.ap50):  # Fixed: check length instead of using index()
        print(f"{name:20} mAP50: {val_results.box.ap50[i]:.4f}")

# Run TESTING
print("\n" + "="*50)
print("RUNNING TESTING ON YGB")
print("="*50)
test_results = model.val(
    data=test_yaml_path,
    split='test',
    imgsz=640,
    batch=16,
    device=0,
    workers=0,
    verbose=True
)

print("\n" + "="*50)
print("YGB TEST RESULTS (nc=4)")
print("="*50)
print(f"mAP50-95: {test_results.box.map:.4f}")
print(f"mAP50:     {test_results.box.map50:.4f}")
print(f"mAP75:     {test_results.box.map75:.4f}")
print(f"Precision: {test_results.box.mp:.4f}")
print(f"Recall:    {test_results.box.mr:.4f}")

# Save results
with open('YGB_COMPLETE_results_nc15.txt', 'w') as f:
    f.write("YGB MODEL - COMPLETE RESULTS (CORRECT nc=4)\n")
    f.write("=" * 50 + "\n\n")
    
    f.write("VALIDATION RESULTS\n")
    f.write("-" * 20 + "\n")
    f.write(f"mAP50-95: {val_results.box.map:.4f}\n")
    f.write(f"mAP50:     {val_results.box.map50:.4f}\n")
    f.write(f"mAP75:     {val_results.box.map75:.4f}\n")
    f.write(f"Precision: {val_results.box.mp:.4f}\n")
    f.write(f"Recall:    {val_results.box.mr:.4f}\n\n")
    
    f.write("TEST RESULTS\n")
    f.write("-" * 20 + "\n")
    f.write(f"mAP50-95: {test_results.box.map:.4f}\n")
    f.write(f"mAP50:     {test_results.box.map50:.4f}\n")
    f.write(f"mAP75:     {test_results.box.map75:.4f}\n")
    f.write(f"Precision: {test_results.box.mp:.4f}\n")
    f.write(f"Recall:    {test_results.box.mr:.4f}\n")


Cleaning old cache files...
✓ Deleted: C:\Users\DELL\Desktop\c\SSL_YGB\val\labels.cache
✓ Deleted: C:\Users\DELL\Desktop\c\SSL_YGB\test\labels.cache
✓ Created: SSL_YGB_val_nc4.yaml
✓ Created: SSL_YGB_test_nc4.yaml

RUNNING VALIDATION ON YGB
Ultralytics 8.4.162  Python-3.13.9 torch-2.7.1+cu118 CUDA:0 (NVIDIA GeForce RTX 2060, 6144MiB)
YOLO26n summary (fused): 120 layers, 2,375,616 parameters, 0 gradients, 5.3 GFLOPs
val: Fast image access  (ping: 0.00.0 ms, read: 2013.21182.2 MB/s, size: 330.2 KB)
val: Scanning C:\Users\DELL\Desktop\c\SSL_YGB\val\labels... 132 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 132/132 2.2Kit/s 0.1s
val: New cache created: C:\Users\DELL\Desktop\c\SSL_YGB\val\labels.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 9/9 4.3it/s 2.1s0.2s
                   all        132       1992      0.647      0.561      0.605      0.292
               class_0         37        367      0.783      0.807   

## LGB  YOLOv26

In [37]:
import torch
import torch.nn as nn
from ultralytics import YOLO
from ultralytics.models.yolo.detect import DetectionTrainer

# ── Same weight computation as above ──────────────────────────────────────────
class_counts = [3927, 2706, 5455, 2948]
N_total      = 15036
C            = 4

raw_weights  = [N_total / (C * n) for n in class_counts]
max_w        = max(raw_weights)                          # ≈ 57.85
norm_weights = [w / max_w for w in raw_weights]

weights_tensor = torch.tensor(norm_weights, dtype=torch.float32)  # shape [4]

# ── Custom Trainer — injects Eq. (8) weighted BCE into YOLO loss ──────────────
class WeightedDetectionTrainer(DetectionTrainer):

    def get_model(self, cfg=None, weights=None, verbose=True):
        model = super().get_model(cfg=cfg, weights=weights, verbose=verbose)

        device = next(model.parameters()).device
        w      = weights_tensor.to(device)              # move to GPU

        # Replace BCEcls with per-class weighted version — implements Eq. (8):
        # L_cls = -Σ w̃_i [ y_i log(ŷ_i) + (1-y_i) log(1-ŷ_i) ]
        if hasattr(model, 'criterion') and model.criterion is not None:
            model.criterion.BCEcls = nn.BCEWithLogitsLoss(
                pos_weight = w,
                reduction  = 'none'
            )
            print(f"✅ Weighted BCE injected | max_w={max_w:.2f} | "
                  f"fighter w̃={norm_weights[9]:.4f} | "
                  f"car w̃={norm_weights[0]:.4f}")
        return model


# ── Training call — same hyperparams as your paper ────────────────────────────
model   = YOLO('yolo26n.pt')    # or 'yolo26n.pt'
model.trainer = WeightedDetectionTrainer

results = model.train(
    data          = 'SSL_LGB/data.yaml',
    epochs        = 100,
    imgsz         = 640,
    batch         = 8,            # paper: batch=8
    lr0           = 0.1,          # paper: lr0=0.1
    optimizer     = 'SGD',
    momentum      = 0.937,
    weight_decay  = 0.0005,
    warmup_epochs = 3,
    cos_lr        = True,         # paper: cosine annealing
    augment       = True,
    pretrained    = False,        # paper: trained from scratch
    patience      = 15,           # paper: early stopping patience=15
    device        = 0,
    project       = 'runs',
    name          = 'SSL_LGB_YOLO26n_weighted',
    exist_ok      = True,
    seed          = 42,
    workers       = 4,
    amp           = False,
    resume        = False,
    cache         = False,
)

print(f"\nBest weights: {results.save_dir}/weights/best.pt")

New https://pypi.org/project/ultralytics/8.4.163 available  Update with 'pip install -U ultralytics'
Ultralytics 8.4.162  Python-3.13.9 torch-2.7.1+cu118 CUDA:0 (NVIDIA GeForce RTX 2060, 6144MiB)
engine\trainer: agnostic_nms=False, amp=False, angle=1.0, augment=True, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=SSL_LGB/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=100, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.1, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo26n.pt,

In [38]:
from ultralytics import YOLO
import yaml
import os
import shutil

# RGB paths
base_path = r"C:\Users\DELL\Desktop\c\SSL_LGB"
model_path = r'C:\Users\DELL\Desktop\c\runs\detect\runs\SSL_LGB_YOLO26n_weighted\weights\best.pt'

# DELETE OLD CACHE FILES
print("Cleaning old cache files...")
cache_files = [
    os.path.join(base_path, "val", "labels.cache"),
    os.path.join(base_path, "test", "labels.cache"),
]
for cache_file in cache_files:
    if os.path.exists(cache_file):
        os.remove(cache_file)
        print(f"✓ Deleted: {cache_file}")

# CORRECT class configuration (4 classes)
num_classes = 4
class_names = ['airplane', 'boat', 'car', 'ship']

# Create VAL YAML
val_yaml_path = 'SSL_LGB_val_nc4.yaml'
val_yaml_content = {
    'path': base_path,
    'train': 'train/images',
    'val': 'VAL/images',
    'test': 'test/images',
    'nc': num_classes,
    'names': class_names,
}
with open(val_yaml_path, 'w') as f:
    yaml.dump(val_yaml_content, f, default_flow_style=False)
print(f"✓ Created: {val_yaml_path}")

# Create TEST YAML
test_yaml_path = 'SSL_LGB_test_nc4.yaml'
test_yaml_content = {
    'path': base_path,
    'train': 'train/images',
    'val': 'VAL/images',
    'test': 'test/images',
    'nc': num_classes,
    'names': class_names,
}
with open(test_yaml_path, 'w') as f:
    yaml.dump(test_yaml_content, f, default_flow_style=False)
print(f"✓ Created: {test_yaml_path}")

# Load model
model = YOLO(model_path)

# Run VALIDATION
print("\n" + "="*50)
print("RUNNING VALIDATION ON LGB")
print("="*50)
val_results = model.val(
    data=val_yaml_path,
    split='val',
    imgsz=640,
    batch=16,
    device=0,
    workers=0,
    verbose=True
)

print("\n" + "="*50)
print("LGB VALIDATION RESULTS (nc=4)")
print("="*50)
print(f"mAP50-95: {val_results.box.map:.4f}")
print(f"mAP50:     {val_results.box.map50:.4f}")
print(f"mAP75:     {val_results.box.map75:.4f}")
print(f"Precision: {val_results.box.mp:.4f}")
print(f"Recall:    {val_results.box.mr:.4f}")

# FIXED PER-CLASS RESULTS
print("\n" + "="*50)
print("PER-CLASS RESULTS (mAP50)")
print("="*50)
for i, name in enumerate(class_names):
    if i < len(val_results.box.ap50):  # Fixed: check length instead of using index()
        print(f"{name:20} mAP50: {val_results.box.ap50[i]:.4f}")

# Run TESTING
print("\n" + "="*50)
print("RUNNING TESTING ON LGB")
print("="*50)
test_results = model.val(
    data=test_yaml_path,
    split='test',
    imgsz=640,
    batch=16,
    device=0,
    workers=0,
    verbose=True
)

print("\n" + "="*50)
print("LGB TEST RESULTS (nc=4)")
print("="*50)
print(f"mAP50-95: {test_results.box.map:.4f}")
print(f"mAP50:     {test_results.box.map50:.4f}")
print(f"mAP75:     {test_results.box.map75:.4f}")
print(f"Precision: {test_results.box.mp:.4f}")
print(f"Recall:    {test_results.box.mr:.4f}")

# Save results
with open('LGB_COMPLETE_results_nc15.txt', 'w') as f:
    f.write("LGB MODEL - COMPLETE RESULTS (CORRECT nc=4)\n")
    f.write("=" * 50 + "\n\n")
    
    f.write("VALIDATION RESULTS\n")
    f.write("-" * 20 + "\n")
    f.write(f"mAP50-95: {val_results.box.map:.4f}\n")
    f.write(f"mAP50:     {val_results.box.map50:.4f}\n")
    f.write(f"mAP75:     {val_results.box.map75:.4f}\n")
    f.write(f"Precision: {val_results.box.mp:.4f}\n")
    f.write(f"Recall:    {val_results.box.mr:.4f}\n\n")
    
    f.write("TEST RESULTS\n")
    f.write("-" * 20 + "\n")
    f.write(f"mAP50-95: {test_results.box.map:.4f}\n")
    f.write(f"mAP50:     {test_results.box.map50:.4f}\n")
    f.write(f"mAP75:     {test_results.box.map75:.4f}\n")
    f.write(f"Precision: {test_results.box.mp:.4f}\n")
    f.write(f"Recall:    {test_results.box.mr:.4f}\n")


Cleaning old cache files...
✓ Deleted: C:\Users\DELL\Desktop\c\SSL_LGB\val\labels.cache
✓ Deleted: C:\Users\DELL\Desktop\c\SSL_LGB\test\labels.cache
✓ Created: SSL_LGB_val_nc4.yaml
✓ Created: SSL_LGB_test_nc4.yaml

RUNNING VALIDATION ON LGB
Ultralytics 8.4.162  Python-3.13.9 torch-2.7.1+cu118 CUDA:0 (NVIDIA GeForce RTX 2060, 6144MiB)
YOLO26n summary (fused): 120 layers, 2,375,616 parameters, 0 gradients, 5.3 GFLOPs
val: Fast image access  (ping: 0.00.0 ms, read: 2300.11572.5 MB/s, size: 336.1 KB)
val: Scanning C:\Users\DELL\Desktop\c\SSL_LGB\val\labels... 132 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 132/132 2.2Kit/s 0.1s
val: New cache created: C:\Users\DELL\Desktop\c\SSL_LGB\val\labels.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 9/9 4.4it/s 2.0s0.2s
                   all        132       1992      0.668      0.573      0.615      0.295
               class_0         37        367        0.8      0.796   